# omicsTL - Transfer Learning Example

<div>
<img src="images/tl_arch1.png" width="500" style="background-color: white;"/>
</div>

This notebook demonstrates a complete transfer-learning workflow using omicsTL. For clarity and reproducibility, we generate linked synthetic source and target datasets from two real viral proteomics datasets. This synthetic-data step is used here to showcase omicsTL’s simulation utilities and to create controlled source/target domain differences.

In practice, you can skip the simulation step and apply the same modeling workflow directly to real source and target datasets with minimal changes.

## Workflow summary

1. Load two real datasets (source and target).

2. Generate linked synthetic datasets with either a continuous or categorical response (for demonstration purposes only-not needed for real data application).

3. Split the synthetic target dataset into train, ensemble, and test partitions.

4. Package everything into a DatasetContainer.

5. Fit transfer learning models (deep learning and random forest variants).

6. Compare transfer learning vs target-only baselines.

# Imports and core utilities
We start by importing the simulation utilities needed to define a response function and generate synthetic data.

In [1]:
import warnings
warnings.filterwarnings("always")

from omicstl.simulation_utils.data_generation import response_function, generate_synth_data

# Load real datasets and define simulation settings
We use two real datasets with aligned features. The first represents the source domain used to train the base model; the second represents the target domain used for adaptation and evaluation.

We also define a non-trivial response function and choose sample sizes and the number of generated features.

In [2]:
import pandas as pd
import numpy as np

source_real_data_path = "data/source_data_real.csv"
target_real_data_path = "data/target_data_real.csv"

source_real_data = pd.read_csv(source_real_data_path, index_col=0)
target_real_data = pd.read_csv(target_real_data_path, index_col=0)

# Non-trivial continuous response function
response_fn = response_function("tanh(df[, 2]) + df[, 1] * df[, ncol(df)] ^ 2")

num_features = 100
num_samples_source = 100
num_samples_target = 50
samples_target_test = 25
samples_target_ensemble = 5

 # Generate linked synthetic datasets (continuous and categorical)

 We generate linked synthetic datasets for both a continuous and categorical response. The critical piece is passing *`prior_lc_info`* from the source simulation into the target simulation. This ensures the source and target synthetic datasets are coupled and reflect a coherent transfer-learning scenario

In [3]:
# Continuous response

source_synth_data_cont, source_lc_info_cont, _ = generate_synth_data(
    data = source_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_source, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 # signal to noise ratio
)

target_synth_data_cont, _, _ = generate_synth_data(
    data = target_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_target + samples_target_test + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    prior_lc_info = source_lc_info_cont, # crucial to include else the source and target datasets aren't linked!
    snr = 1 # signal to noise ratio
)

display(source_synth_data_cont)
display(target_synth_data_cont)

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-4.178608,-0.936221,-3.131792,3.963236,1.879877,1.073573,0.160352,6.243121,4.421815,0.850644,...,2.516906,7.244463,3.494390,-0.604225,2.722824,7.025628,0.605976,7.285271,-2.648455,-1.854619
1,-4.082730,-0.817488,-5.357407,3.983718,1.999953,1.188203,-0.655431,6.436111,4.515150,0.773420,...,1.195323,7.054645,3.582885,-0.566436,2.963858,7.301699,0.609052,6.592369,-2.392537,-1.589228
2,-1.910832,-0.796906,-4.470238,3.940995,2.317394,1.778970,0.103470,6.543203,4.410021,0.690011,...,1.359478,7.001954,3.720259,-0.667006,2.843733,7.235559,0.753414,6.881722,-3.064396,-0.273899
3,-4.196538,-0.875468,-4.581824,3.738317,2.037673,1.231108,-0.178174,6.322429,4.629310,0.757515,...,2.605575,7.149532,3.590919,-0.678520,2.675125,7.049058,0.496084,6.581750,-1.940747,-0.978709
4,2.716900,-0.767548,-2.967484,3.530678,1.596846,1.463723,0.204698,6.495668,4.386031,0.741122,...,1.361245,7.084282,3.673621,-0.613607,2.765257,7.178792,0.603109,6.660294,-2.583702,0.222263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-3.379824,-0.743143,-3.407787,4.242248,2.129859,1.247652,-1.048910,6.406783,4.758773,0.828025,...,1.951700,7.168689,3.681548,-0.601433,2.824414,7.222368,0.674505,7.153285,-1.836669,-1.761468
96,-7.802230,-0.828193,-3.215061,3.375386,1.700152,1.669604,0.189081,6.684461,4.490309,0.794492,...,2.677325,6.848100,3.649122,-0.763130,2.610764,7.259708,0.602915,7.256666,-2.565829,-2.441064
97,-2.220610,-0.846516,-3.872839,3.965743,1.682274,1.776423,-1.399248,6.380188,3.626153,0.773170,...,2.010842,7.087619,3.717460,-0.632831,2.506682,7.396047,0.610073,7.054944,-2.441091,-0.746728
98,-3.403293,-0.989964,-4.215553,3.339681,2.050608,1.786808,-0.383068,6.299359,3.890109,0.635865,...,1.847615,7.005142,3.694262,-0.677674,2.736813,7.146987,0.694534,6.924527,-2.684639,-1.185892


,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-4.541650,-0.890448,-4.621962,4.095663,2.124645,1.926682,-0.112637,6.438194,4.410402,0.792570,...,1.455793,7.108027,3.413638,-0.698168,2.793339,7.296421,0.622742,6.954228,-2.426621,-1.838866
1,-6.530702,-0.858790,-3.162461,3.613274,2.116944,1.578841,-0.089005,6.213861,4.130918,0.755086,...,1.856991,6.694987,3.201516,-0.681852,2.574661,7.304460,0.674289,7.342745,-2.186227,-2.262043
2,-2.152769,-0.846964,-4.221919,3.759522,1.990755,0.908481,0.273516,6.601566,4.197202,0.871468,...,3.353025,7.022967,3.492845,-0.725194,2.605228,7.137887,0.871001,7.289125,-1.745837,-1.905901
3,0.046238,-0.848599,-3.391938,3.778581,1.657491,1.969303,-1.303604,6.679540,4.743427,0.747640,...,2.280191,7.118840,3.281747,-0.539692,2.295358,7.277619,0.477360,6.777718,-2.205024,-0.652453
4,-5.123437,-0.855354,-4.573094,3.401954,2.066323,0.743949,0.476356,6.369472,4.070933,0.735039,...,1.953807,7.183134,3.215023,-0.526330,2.288949,7.093472,0.642211,7.041344,-2.331920,-1.849325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,-6.039350,-0.871934,-4.758069,3.948799,1.747749,1.042233,-0.839037,6.583662,4.636657,0.843838,...,0.985191,7.015234,3.573824,-0.672183,2.718360,7.278487,0.549920,7.311899,-1.726113,-2.051194
76,-5.576799,-0.827457,-3.831877,3.389488,2.246536,2.144375,-0.422834,6.008965,3.732579,0.798796,...,1.568415,7.036590,3.267211,-0.596210,2.706344,7.389998,0.521363,7.570336,-1.929035,-1.877592
77,0.695477,-0.966921,-4.675436,3.734654,1.648910,1.520412,-0.606715,6.402170,3.379535,0.648670,...,1.763112,6.972667,3.481586,-0.656739,2.921403,7.166421,0.543226,6.458017,-1.758109,-0.370065
78,1.478610,-0.929538,-3.023475,3.035966,1.777125,1.814620,-1.109275,6.129313,3.861876,0.839396,...,2.121201,7.059073,3.616462,-0.730867,2.195539,7.474186,0.569468,6.971418,-1.032729,-0.571058


In [4]:
# Categorical response

source_synth_data_cat, source_lc_info_cat, _ = generate_synth_data(
    data = source_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_source + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    snr = 1, # signal to noise ratio
    response_parameters={
        "ncats": 3,
        "quantile": "quantile"
    }
)

target_synth_data_cat, source_lc_info_cat, _ = generate_synth_data(
    data = target_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_target + samples_target_test + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 ,# signal to noise ratio
    prior_lc_info = source_lc_info_cat, # crucial to include else the source and target datasets aren't linked!
    response_parameters={
        "ncats": 3,
        "quantile": "quantile"
    }
)

display(source_synth_data_cat)
display(target_synth_data_cat)

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,2.0,0.334398,-0.530747,3.519993,1.581034,2.769817,5.696282,3.024011,-0.391179,1.218864,...,-1.298991,0.291674,-0.941151,1.170355,0.416906,0.440237,-0.422287,3.653889,2.637006,0.637548
1,1.0,0.556218,-0.664416,3.192342,1.211805,2.821828,6.080413,2.940917,-0.476082,1.456385,...,-0.040265,0.499437,-1.063773,1.202782,0.159295,0.164896,0.151300,3.895463,2.744405,-0.416676
2,2.0,0.493302,-0.239337,3.126945,1.567505,3.329710,5.314973,2.748524,-0.384308,1.481552,...,-1.935438,0.630709,-1.288975,1.400199,0.220011,0.423452,-0.825965,4.073053,2.995863,0.071110
3,3.0,0.681626,-0.600280,1.874606,1.704365,3.555062,5.205216,3.050447,-0.533948,1.152309,...,-1.182547,0.809154,-1.236513,1.505007,-0.165080,0.318150,-0.738560,3.828871,3.006598,-0.420872
4,3.0,0.810447,-0.341681,3.793977,1.819370,3.601603,5.618282,3.046022,-0.364164,1.118525,...,-1.377611,0.477185,-1.275298,1.376691,0.679716,0.004405,-0.313425,3.725643,2.596991,1.258176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,2.0,0.870678,-1.339982,2.558049,1.819975,2.107324,5.348888,3.063792,-0.288935,1.817470,...,-1.360374,1.035291,-1.231257,1.461814,0.370117,0.708456,-0.376733,4.112995,2.869435,0.766181
101,2.0,0.780412,-0.988387,3.086696,1.243490,3.341254,5.555080,2.742807,-0.647924,1.434731,...,-0.850647,0.116357,-1.141891,1.144026,0.313134,0.237738,-0.931490,3.153645,2.527246,-0.464861
102,1.0,0.688295,-1.635881,3.347342,1.823465,2.893440,5.513496,3.012955,-0.452252,1.208032,...,-0.828807,0.025309,-1.351444,0.959442,0.542103,0.350006,-0.665100,4.166639,2.713337,0.121243
103,1.0,0.902209,-0.667869,2.402830,1.393846,4.274411,5.555013,2.681408,-0.612985,1.659035,...,-0.843363,-0.340832,-0.958770,0.772364,0.025402,0.502419,0.005468,3.776576,2.739144,0.256146


,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,1.0,0.552518,-1.165635,2.932732,1.821936,3.128168,5.801607,3.029587,-0.494024,1.446902,...,-1.575824,0.914423,-0.948219,1.624394,0.023860,0.403502,-0.229997,4.048471,2.862950,0.763344
1,2.0,0.400899,-1.608266,2.191020,1.283271,2.418339,5.582597,2.878881,-0.491202,1.414338,...,-1.694914,0.157998,-1.344537,1.234787,0.238244,0.566472,-0.262567,4.096030,2.932571,0.052959
2,3.0,0.592400,-0.532348,2.284822,1.803134,4.412973,5.665747,3.085942,-0.437651,1.599014,...,-0.171198,0.175333,-0.973109,0.948464,0.146575,0.338169,-0.180829,4.919150,2.783840,-0.314072
3,2.0,0.555483,-0.826420,2.683675,2.139698,3.140517,5.489577,2.845138,-0.221734,1.406254,...,-0.691968,0.598545,-1.207544,1.071407,0.132470,0.281845,-0.261720,4.230703,2.900352,0.221365
4,1.0,1.392967,-0.624720,1.004457,1.765839,4.196910,5.948934,2.834068,-0.418774,1.023580,...,-0.936572,0.035990,-1.040006,1.478505,0.234338,0.093808,-0.972741,4.603550,3.034843,0.503189
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2.0,0.470939,-0.745384,2.557317,1.386710,3.194124,5.268439,2.946649,-0.293810,1.183338,...,-1.478531,1.020161,-1.326696,1.328028,-0.267671,0.478990,0.162855,3.791275,2.711117,0.875482
76,1.0,0.975923,-0.971192,2.126422,1.086101,3.662870,5.049236,2.908231,-0.376866,1.166317,...,-1.913286,0.742031,-1.361269,0.872956,-0.328305,0.257805,0.272319,4.265762,2.736400,0.360599
77,3.0,1.089183,-0.294300,2.917683,1.356332,3.954985,5.601494,3.122140,-0.259882,1.544301,...,-1.190939,0.541190,-0.947073,1.032900,0.042658,0.097765,-0.382884,4.289828,2.917302,0.598062
78,2.0,0.807530,-0.063545,1.961713,1.272109,4.360372,6.009302,3.270187,-0.379602,1.398175,...,-0.706061,0.685784,-1.229169,1.067236,0.296798,0.626619,-0.786527,3.822305,2.565929,0.740770


# Standardize response column naming
We rename the response column to "response" for consistency across model fitting, and we cast the categorical response to integer labels

In [5]:
# Set response column name

source_synth_data_cont.rename(columns={source_synth_data_cont.columns[0]: 'response'}, inplace=True)
target_synth_data_cont.rename(columns={target_synth_data_cont.columns[0]: 'response'}, inplace=True)
display(source_synth_data_cont)
display(target_synth_data_cont)

source_synth_data_cat.rename(columns={source_synth_data_cat.columns[0]: 'response'}, inplace=True)
target_synth_data_cat.rename(columns={target_synth_data_cat.columns[0]: 'response'}, inplace=True)
source_synth_data_cat = source_synth_data_cat.astype({"response": np.int64})
target_synth_data_cat = target_synth_data_cat.astype({"response": np.int64})
display(source_synth_data_cat)
display(target_synth_data_cat)

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-4.178608,-0.936221,-3.131792,3.963236,1.879877,1.073573,0.160352,6.243121,4.421815,0.850644,...,2.516906,7.244463,3.494390,-0.604225,2.722824,7.025628,0.605976,7.285271,-2.648455,-1.854619
1,-4.082730,-0.817488,-5.357407,3.983718,1.999953,1.188203,-0.655431,6.436111,4.515150,0.773420,...,1.195323,7.054645,3.582885,-0.566436,2.963858,7.301699,0.609052,6.592369,-2.392537,-1.589228
2,-1.910832,-0.796906,-4.470238,3.940995,2.317394,1.778970,0.103470,6.543203,4.410021,0.690011,...,1.359478,7.001954,3.720259,-0.667006,2.843733,7.235559,0.753414,6.881722,-3.064396,-0.273899
3,-4.196538,-0.875468,-4.581824,3.738317,2.037673,1.231108,-0.178174,6.322429,4.629310,0.757515,...,2.605575,7.149532,3.590919,-0.678520,2.675125,7.049058,0.496084,6.581750,-1.940747,-0.978709
4,2.716900,-0.767548,-2.967484,3.530678,1.596846,1.463723,0.204698,6.495668,4.386031,0.741122,...,1.361245,7.084282,3.673621,-0.613607,2.765257,7.178792,0.603109,6.660294,-2.583702,0.222263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-3.379824,-0.743143,-3.407787,4.242248,2.129859,1.247652,-1.048910,6.406783,4.758773,0.828025,...,1.951700,7.168689,3.681548,-0.601433,2.824414,7.222368,0.674505,7.153285,-1.836669,-1.761468
96,-7.802230,-0.828193,-3.215061,3.375386,1.700152,1.669604,0.189081,6.684461,4.490309,0.794492,...,2.677325,6.848100,3.649122,-0.763130,2.610764,7.259708,0.602915,7.256666,-2.565829,-2.441064
97,-2.220610,-0.846516,-3.872839,3.965743,1.682274,1.776423,-1.399248,6.380188,3.626153,0.773170,...,2.010842,7.087619,3.717460,-0.632831,2.506682,7.396047,0.610073,7.054944,-2.441091,-0.746728
98,-3.403293,-0.989964,-4.215553,3.339681,2.050608,1.786808,-0.383068,6.299359,3.890109,0.635865,...,1.847615,7.005142,3.694262,-0.677674,2.736813,7.146987,0.694534,6.924527,-2.684639,-1.185892


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-4.541650,-0.890448,-4.621962,4.095663,2.124645,1.926682,-0.112637,6.438194,4.410402,0.792570,...,1.455793,7.108027,3.413638,-0.698168,2.793339,7.296421,0.622742,6.954228,-2.426621,-1.838866
1,-6.530702,-0.858790,-3.162461,3.613274,2.116944,1.578841,-0.089005,6.213861,4.130918,0.755086,...,1.856991,6.694987,3.201516,-0.681852,2.574661,7.304460,0.674289,7.342745,-2.186227,-2.262043
2,-2.152769,-0.846964,-4.221919,3.759522,1.990755,0.908481,0.273516,6.601566,4.197202,0.871468,...,3.353025,7.022967,3.492845,-0.725194,2.605228,7.137887,0.871001,7.289125,-1.745837,-1.905901
3,0.046238,-0.848599,-3.391938,3.778581,1.657491,1.969303,-1.303604,6.679540,4.743427,0.747640,...,2.280191,7.118840,3.281747,-0.539692,2.295358,7.277619,0.477360,6.777718,-2.205024,-0.652453
4,-5.123437,-0.855354,-4.573094,3.401954,2.066323,0.743949,0.476356,6.369472,4.070933,0.735039,...,1.953807,7.183134,3.215023,-0.526330,2.288949,7.093472,0.642211,7.041344,-2.331920,-1.849325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,-6.039350,-0.871934,-4.758069,3.948799,1.747749,1.042233,-0.839037,6.583662,4.636657,0.843838,...,0.985191,7.015234,3.573824,-0.672183,2.718360,7.278487,0.549920,7.311899,-1.726113,-2.051194
76,-5.576799,-0.827457,-3.831877,3.389488,2.246536,2.144375,-0.422834,6.008965,3.732579,0.798796,...,1.568415,7.036590,3.267211,-0.596210,2.706344,7.389998,0.521363,7.570336,-1.929035,-1.877592
77,0.695477,-0.966921,-4.675436,3.734654,1.648910,1.520412,-0.606715,6.402170,3.379535,0.648670,...,1.763112,6.972667,3.481586,-0.656739,2.921403,7.166421,0.543226,6.458017,-1.758109,-0.370065
78,1.478610,-0.929538,-3.023475,3.035966,1.777125,1.814620,-1.109275,6.129313,3.861876,0.839396,...,2.121201,7.059073,3.616462,-0.730867,2.195539,7.474186,0.569468,6.971418,-1.032729,-0.571058


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,2,0.334398,-0.530747,3.519993,1.581034,2.769817,5.696282,3.024011,-0.391179,1.218864,...,-1.298991,0.291674,-0.941151,1.170355,0.416906,0.440237,-0.422287,3.653889,2.637006,0.637548
1,1,0.556218,-0.664416,3.192342,1.211805,2.821828,6.080413,2.940917,-0.476082,1.456385,...,-0.040265,0.499437,-1.063773,1.202782,0.159295,0.164896,0.151300,3.895463,2.744405,-0.416676
2,2,0.493302,-0.239337,3.126945,1.567505,3.329710,5.314973,2.748524,-0.384308,1.481552,...,-1.935438,0.630709,-1.288975,1.400199,0.220011,0.423452,-0.825965,4.073053,2.995863,0.071110
3,3,0.681626,-0.600280,1.874606,1.704365,3.555062,5.205216,3.050447,-0.533948,1.152309,...,-1.182547,0.809154,-1.236513,1.505007,-0.165080,0.318150,-0.738560,3.828871,3.006598,-0.420872
4,3,0.810447,-0.341681,3.793977,1.819370,3.601603,5.618282,3.046022,-0.364164,1.118525,...,-1.377611,0.477185,-1.275298,1.376691,0.679716,0.004405,-0.313425,3.725643,2.596991,1.258176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,2,0.870678,-1.339982,2.558049,1.819975,2.107324,5.348888,3.063792,-0.288935,1.817470,...,-1.360374,1.035291,-1.231257,1.461814,0.370117,0.708456,-0.376733,4.112995,2.869435,0.766181
101,2,0.780412,-0.988387,3.086696,1.243490,3.341254,5.555080,2.742807,-0.647924,1.434731,...,-0.850647,0.116357,-1.141891,1.144026,0.313134,0.237738,-0.931490,3.153645,2.527246,-0.464861
102,1,0.688295,-1.635881,3.347342,1.823465,2.893440,5.513496,3.012955,-0.452252,1.208032,...,-0.828807,0.025309,-1.351444,0.959442,0.542103,0.350006,-0.665100,4.166639,2.713337,0.121243
103,1,0.902209,-0.667869,2.402830,1.393846,4.274411,5.555013,2.681408,-0.612985,1.659035,...,-0.843363,-0.340832,-0.958770,0.772364,0.025402,0.502419,0.005468,3.776576,2.739144,0.256146


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,1,0.552518,-1.165635,2.932732,1.821936,3.128168,5.801607,3.029587,-0.494024,1.446902,...,-1.575824,0.914423,-0.948219,1.624394,0.023860,0.403502,-0.229997,4.048471,2.862950,0.763344
1,2,0.400899,-1.608266,2.191020,1.283271,2.418339,5.582597,2.878881,-0.491202,1.414338,...,-1.694914,0.157998,-1.344537,1.234787,0.238244,0.566472,-0.262567,4.096030,2.932571,0.052959
2,3,0.592400,-0.532348,2.284822,1.803134,4.412973,5.665747,3.085942,-0.437651,1.599014,...,-0.171198,0.175333,-0.973109,0.948464,0.146575,0.338169,-0.180829,4.919150,2.783840,-0.314072
3,2,0.555483,-0.826420,2.683675,2.139698,3.140517,5.489577,2.845138,-0.221734,1.406254,...,-0.691968,0.598545,-1.207544,1.071407,0.132470,0.281845,-0.261720,4.230703,2.900352,0.221365
4,1,1.392967,-0.624720,1.004457,1.765839,4.196910,5.948934,2.834068,-0.418774,1.023580,...,-0.936572,0.035990,-1.040006,1.478505,0.234338,0.093808,-0.972741,4.603550,3.034843,0.503189
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2,0.470939,-0.745384,2.557317,1.386710,3.194124,5.268439,2.946649,-0.293810,1.183338,...,-1.478531,1.020161,-1.326696,1.328028,-0.267671,0.478990,0.162855,3.791275,2.711117,0.875482
76,1,0.975923,-0.971192,2.126422,1.086101,3.662870,5.049236,2.908231,-0.376866,1.166317,...,-1.913286,0.742031,-1.361269,0.872956,-0.328305,0.257805,0.272319,4.265762,2.736400,0.360599
77,3,1.089183,-0.294300,2.917683,1.356332,3.954985,5.601494,3.122140,-0.259882,1.544301,...,-1.190939,0.541190,-0.947073,1.032900,0.042658,0.097765,-0.382884,4.289828,2.917302,0.598062
78,2,0.807530,-0.063545,1.961713,1.272109,4.360372,6.009302,3.270187,-0.379602,1.398175,...,-0.706061,0.685784,-1.229169,1.067236,0.296798,0.626619,-0.786527,3.822305,2.565929,0.740770


# Split target into train, ensemble, and test

* We split the target dataset into:

* train: used for target-side fitting/adaptation

* ensemble: a small held-out target partition used to estimate performance-based weights for combining individual random forest models into a weighted ensemble

* test: final evaluation set

For categorical responses, we stratify splits to ensure each partition contains all classes.

In [6]:
from sklearn.model_selection import train_test_split

target_synth_train_comb_cont, target_synth_test_cont = train_test_split(
    target_synth_data_cont,
    train_size = num_samples_target + samples_target_ensemble,
    test_size = samples_target_test
)

target_synth_train_comb_cat, target_synth_test_cat = train_test_split(
    target_synth_data_cat,
    train_size = num_samples_target + samples_target_ensemble,
    test_size = samples_target_test,
    stratify = target_synth_data_cat['response'] # crucial or else random forest might break due to partition not representing all classes.
)

target_synth_train_cont, target_synth_train_ensemble_cont = train_test_split(
    target_synth_train_comb_cont,
    train_size = num_samples_target,
    test_size = samples_target_ensemble
)

target_synth_train_cat, target_synth_train_ensemble_cat = train_test_split(
    target_synth_train_comb_cat,
    train_size = num_samples_target,
    test_size = samples_target_ensemble,
    stratify = target_synth_train_comb_cat['response'] # crucial or else random forest might break due to partition not representing all classes.
)

# Create DatasetContainer objects
We provide a DatasetContainer helper class to keep the different datasets organized. Optionally, `id_tuple` can also be set which is used to label the scenario and replicate IDs if performing large scale simulation studies. These IDs are passed to the output after model fitting.

In [7]:
from omicstl.simulation_utils.data_utils import DatasetContainer
datasets_cont = DatasetContainer(
	source_data=source_synth_data_cont,
	target_data=target_synth_train_cont,
    target_ensemble_data=target_synth_train_ensemble_cont,
	target_test_data=[target_synth_test_cont]
)
datasets_cont.set_response_column("response") # Identify response column

datasets_cat = DatasetContainer(
	source_data=source_synth_data_cat,
	target_data=target_synth_train_cat,
    target_ensemble_data=target_synth_train_ensemble_cat,
	target_test_data=[target_synth_test_cat]
)
datasets_cat.set_response_column("response") # Identify response column

# Fit transfer-learning models (deep learning and random forest)

We demonstrate both deep learning and random-forest transfer learning models. Deep learning models support automatic tuning over a parameter grid via *fit_dl_model*. Random forest models are fit through a Python interface to the R implementation

In [8]:
param_grid = {
	"dropout": [0.25, 0.5],
	"n_latent_dims": [2],
	"hidden_dim_base": [6],
	"lr": [0.01, 0.001],
	"source_epochs": [1000],
	"target_epochs": [1000],
	"freeze": ["none"],
	"weight_decay": [1e-4, 1e-2],
	"gamma": [1, 2, 3]
}

Deep learning models can be fit using a DatasetContainer and parameter grid using `fit_dl_model`, which returns a dataframe with results, the trained transfer learning model, and the model trained on only the target dataset.

In [9]:
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model

random.seed(42)
torch.manual_seed(42)
out, mult_vae_cont_model, model_targetonly = fit_dl_model(
	datasets_cont,
	"mult_vae",
	device("cpu"),
	param_grid
)
display(out)

random.seed(42)
torch.manual_seed(42)
out, mult_vae_cat_model, model_targetonly = fit_dl_model(
	datasets_cat,
	"mult_vae",
	device("cpu"),
	param_grid
)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,2.299964,1.928906,NaN,NaN,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,2.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,2.679919,2.048420,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,2.0


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavi

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,NaN,NaN,0.36,0.330928,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.01,3.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,NaN,NaN,0.36,0.190588,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.01,1.0


In [10]:
random.seed(42)
torch.manual_seed(42)
out, mult_mlp_cont_model, model_targetonly = fit_dl_model(
	datasets_cont,
	"mult_mlp",
	device("cpu"),
	param_grid
)
display(out)

random.seed(42)
torch.manual_seed(42)
out, mult_mlp_cat_model, model_targetonly = fit_dl_model(
	datasets_cat,
	"mult_mlp",
	device("cpu"),
	param_grid
)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,2.118096,1.696145,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0100,1.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,2.044846,1.609328,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,3.0


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavi

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,NaN,NaN,0.36,0.347431,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,2.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,NaN,NaN,0.44,0.457355,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,2.0


The random forest based models can also be fit using a python interface to the R modeling code.

In [11]:
from omicstl.r_utils import set_seed
from omicstl.simulation_utils.model_utils import fit_rf_model

random.seed(42)
out, rf_cont_model = fit_rf_model(datasets_cont)
display(out)

random.seed(42)
set_seed(42)
out, rf_cat_model = fit_rf_model(datasets_cat)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1238: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1238: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,2.569616,2.059785,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,test_0,rf,pred_source_full_val,target,2.569616,2.059785,NaN,NaN,NaN,NaN,NaN,NaN
2,None,None,test_0,rf,pred_0_full,target,2.397624,1.911197,NaN,NaN,NaN,NaN,NaN,NaN
3,None,None,test_0,rf,pred_0_full_val,target,2.397624,1.911197,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,test_0,rf,pred_1_full,target,2.055868,1.767553,NaN,NaN,NaN,NaN,NaN,NaN
5,None,None,test_0,rf,pred_1_full_val,target,2.055868,1.767553,NaN,NaN,NaN,NaN,NaN,NaN
6,None,None,test_0,rf,pred_2_full,target,2.181441,1.811991,NaN,NaN,NaN,NaN,NaN,NaN
7,None,None,test_0,rf,pred_2_full_val,target,2.181441,1.811991,NaN,NaN,NaN,NaN,NaN,NaN
8,None,None,test_0,rf,pred_3_full,target,2.311168,1.876771,NaN,NaN,NaN,NaN,NaN,NaN
9,None,None,test_0,rf,pred_3_full_val,target,2.311168,1.876771,NaN,NaN,NaN,NaN,NaN,NaN


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1238: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1238: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1238: F

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,NaN,NaN,0.36,0.348522,0.028268,0.380000,0.36,0.501176
1,None,None,test_0,rf,pred_source_full_val,target,NaN,NaN,0.36,0.348522,0.028268,0.380000,0.36,0.501176
2,None,None,test_0,rf,pred_0_full,target,NaN,NaN,0.28,0.263714,-0.078842,0.262857,0.28,0.493162
3,None,None,test_0,rf,pred_0_full_val,target,NaN,NaN,0.28,0.263714,-0.078842,0.262857,0.28,0.493162
4,None,None,test_0,rf,pred_1_full,target,NaN,NaN,0.32,0.321778,-0.027380,0.354909,0.32,0.557206
5,None,None,test_0,rf,pred_1_full_val,target,NaN,NaN,0.32,0.321778,-0.027380,0.354909,0.32,0.557206
6,None,None,test_0,rf,pred_2_full,target,NaN,NaN,0.32,0.284522,-0.035977,0.265714,0.32,0.516029
7,None,None,test_0,rf,pred_2_full_val,target,NaN,NaN,0.32,0.284522,-0.035977,0.265714,0.32,0.516029
8,None,None,test_0,rf,pred_3_full,target,NaN,NaN,0.36,0.316522,0.028584,0.288571,0.36,0.498750
9,None,None,test_0,rf,pred_3_full_val,target,NaN,NaN,0.36,0.316522,0.028584,0.288571,0.36,0.498750


# Predicting data using a pretrained transfer learning model

Once you have created a model, you can then use it to predict response values for novel inputs.
For deep learning models you can use the `predict_dl_model` function.

In [12]:
from omicstl.simulation_utils.model_utils import predict_dl_model

# Create "new" data by shuffling existing values
new_synth_data_cont = source_synth_data_cont.iloc[:, 1:]
values = new_synth_data_cont.values.flatten()
np.random.shuffle(values)
new_synth_data_cont = pd.DataFrame(
    values.reshape(new_synth_data_cont.shape),
    columns=new_synth_data_cont.columns
)

new_synth_data_cat = source_synth_data_cat.iloc[:, 1:]
values = new_synth_data_cat.values.flatten()
np.random.shuffle(values)
new_synth_data_cat = pd.DataFrame(
    values.reshape(new_synth_data_cat.shape),
    columns=new_synth_data_cat.columns
)

# Get predicted response values on the new data
out = predict_dl_model(mult_vae_cont_model, new_synth_data_cont)
display(out)

out = predict_dl_model(mult_vae_cat_model, new_synth_data_cat)
display(out)

out = predict_dl_model(mult_mlp_cont_model, new_synth_data_cont)
display(out)

out = predict_dl_model(mult_mlp_cat_model, new_synth_data_cat)
display(out)

array([-1.0188333e+00,  2.9566526e-02, -1.2181775e+01, -2.3068309e-01,
        1.7267394e-01, -2.1686792e-02,  1.1512756e-02, -1.9685745e-02,
       -4.1602862e-01, -2.6570805e+01, -2.8887587e+00, -4.3325806e-01,
       -1.0975184e+00, -1.6360903e-01, -3.0964956e+00,  7.3282719e-03,
       -5.1382065e-02, -1.7832541e-01, -4.5629096e-01,  7.5445175e-02,
       -9.0036135e+00, -1.0920174e+01, -1.7251732e+01, -3.3570814e+00,
       -2.3122859e-01, -5.8189735e+00, -6.0959101e-02,  2.9797554e-02,
       -1.1198345e+00, -7.1865625e+00, -8.4146652e+00, -1.4335799e-01,
       -4.7278488e-01, -1.1423320e+01, -1.7459362e+01, -2.7628994e-01,
       -1.1148086e+01, -2.1150804e-01, -1.8423044e+01, -1.3622985e+00,
       -2.1696369e+01,  4.0656805e-02,  3.6463499e-02, -3.4988666e-01,
        5.4377794e-02, -3.2571888e-01, -2.4780345e-01, -4.8465967e-02,
       -1.2833762e-01,  3.6463499e-02, -1.0833937e+01, -4.2042351e-01,
        3.6463499e-02,  3.6463499e-02, -2.6234651e-01, -6.0831273e-01,
      

array([1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1,
       0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 2, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 2, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1])

array([-3.684332  , -2.9421258 , -0.14318001,  1.9229718 ,  1.5766844 ,
       -1.5810281 , -1.8545566 , -1.6252139 , -1.5108095 , -2.662136  ,
       -1.5390177 , -1.5197778 , -2.7036524 , -1.4986819 , -2.7538943 ,
       -0.04440463, -1.4340459 , -2.3729308 , -2.2953563 , -0.02867174,
       -4.7263856 , -1.7308202 , -6.878878  , -3.0764716 , -2.2704039 ,
        0.5456859 , -1.8693689 , -2.5893893 , -3.6972852 , -2.049069  ,
       -6.6014776 ,  0.5928458 , -0.14318001, -3.2872446 , -7.3787603 ,
       -3.0536804 , -2.1055765 , -2.594799  , -4.665086  ,  0.59613574,
       -8.976208  ,  0.23064661, -0.28876054, -1.5840517 ,  0.04862118,
       -0.08836114, -0.14318001, -3.5399323 , -5.5166726 , -0.14318001,
       -5.4786577 ,  1.5727569 ,  4.521847  , -1.044934  ,  2.0908818 ,
        1.6985356 , -6.624966  ,  0.31839418, -0.14318001,  1.0451652 ,
       -3.2426481 , -3.1741657 , -1.7208868 , -0.14318001, -2.5090945 ,
        5.6974535 , -0.14318001, -1.1637161 , -2.7193813 , -0.14

array([2, 1, 1, 1, 2, 2, 2, 1, 0, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2,
       0, 2, 2, 1, 1, 0, 2, 1, 1, 1, 0, 2, 1, 2, 1, 2, 2, 1, 2, 0, 0, 1,
       0, 2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 1, 2, 2, 1, 1, 0, 1, 0, 2,
       2, 2, 2, 1, 1, 1, 2, 1, 2, 0, 0, 2, 0, 0, 2, 2, 2, 2, 2, 0, 1, 2,
       1, 0, 2, 1, 2, 0, 1, 1, 0, 2, 1, 1, 2, 2, 2, 2, 1])

Similarly, you can use the `predict_rf_model` function to predict response values using a random forest model. Random forest predictions will include the predicted responses of each of the transfer learning methods. You can select the appropriate method by observing the reported accuracies of each method from the original `fit_rf_model` call.

In [13]:
from omicstl.simulation_utils.model_utils import predict_rf_model

out = predict_rf_model(rf_cont_model, new_synth_data_cont)
display(pd.DataFrame(out))

out = predict_rf_model(rf_cat_model, new_synth_data_cat)
display(pd.DataFrame(out))

,pred_source,pred_0,pred_1,pred_2,pred_3,pred_ensemble
0,-5.201977,-2.405523,-1.932764,-4.157264,-2.703252,-2.799701
1,-3.050513,-2.284227,-1.567782,-2.336092,-1.725279,-1.978345
2,-3.937549,-2.541339,-1.665000,-2.862005,-2.130543,-2.299722
3,-3.448298,-2.158539,-1.910623,-2.682807,-1.838822,-2.147698
4,-3.080027,-2.494358,-1.919273,-2.503762,-1.853990,-2.192846
...,...,...,...,...,...,...
95,-4.795883,-2.214974,-1.863785,-3.806950,-2.567532,-2.613310
96,-2.984356,-2.179540,-1.630060,-1.625204,-1.730429,-1.791308
97,-3.594112,-2.292458,-1.829423,-2.534638,-1.862043,-2.129640
98,-3.164940,-2.458049,-2.111753,-2.610234,-1.914626,-2.273666


,pred_source,pred_source_prob_1,pred_source_prob_2,pred_source_prob_3,pred_0,pred_0_prob_1,pred_0_prob_2,pred_0_prob_3,pred_1,pred_1_prob_1,...,pred_2_prob_2,pred_2_prob_3,pred_3,pred_3_prob_1,pred_3_prob_2,pred_3_prob_3,pred_ensemble,pred_ensemble_prob_1,pred_ensemble_prob_2,pred_ensemble_prob_3
0,3,0.286,0.298,0.416,2,0.266,0.408,0.326,2,0.206,...,0.346036,0.399965,2,0.276,0.420,0.304,2,0.087533,0.589113,0.323354
1,2,0.316,0.388,0.296,3,0.326,0.332,0.342,2,0.332,...,0.355695,0.320262,2,0.330,0.358,0.312,2,0.311621,0.384742,0.303637
2,2,0.330,0.362,0.308,2,0.294,0.372,0.334,2,0.274,...,0.367408,0.351218,2,0.312,0.404,0.284,2,0.177880,0.538884,0.283236
3,2,0.302,0.396,0.302,2,0.258,0.408,0.334,3,0.172,...,0.379594,0.297442,2,0.288,0.450,0.262,2,0.095281,0.649960,0.254759
4,3,0.196,0.342,0.462,3,0.278,0.340,0.382,3,0.094,...,0.361243,0.434312,3,0.230,0.358,0.412,3,0.022597,0.291092,0.686311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,2,0.254,0.404,0.342,3,0.314,0.302,0.384,3,0.174,...,0.374547,0.365356,2,0.304,0.354,0.342,3,0.102665,0.224576,0.672759
101,1,0.452,0.350,0.198,1,0.362,0.316,0.322,1,0.376,...,0.335187,0.259514,1,0.376,0.332,0.292,1,0.529756,0.265840,0.204404
102,2,0.230,0.418,0.352,2,0.266,0.370,0.364,3,0.102,...,0.404089,0.363113,2,0.212,0.420,0.368,3,0.027417,0.347152,0.625430
103,2,0.316,0.418,0.266,2,0.292,0.354,0.354,3,0.262,...,0.363957,0.309155,2,0.342,0.396,0.262,2,0.227049,0.482179,0.290773


# Advanced usage
Advanced users can also work directly with the base classes we provide for each TL model via TransferForest() and MultiViewModel(). 